# 🇲🇲 AI Voice Studio — VoxCPM2 Myanmar Voice Clone
Phone-only test. Open this notebook in Google Colab from your phone, enable a GPU runtime, then run the cells. The model is downloaded to the Colab session and is not stored in GitHub.

**Model:** `openbmb/VoxCPM2` • Burmese/Myanmar • 2B parameters • about 5 GB model download.

Free Colab availability and runtime limits are controlled by Google and can change; this notebook does not impose a paid API generation quota.

In [ ]:
!pip -q install -U voxcpm gradio librosa soundfile
import torch
print('Torch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM GB:', round(torch.cuda.get_device_properties(0).total_memory/1024**3, 2))
else:
    raise RuntimeError('GPU is required for a practical VoxCPM2 phone-only test. In Colab choose Runtime > Change runtime type > GPU.')

In [ ]:
from voxcpm import VoxCPM
import numpy as np
import librosa
import soundfile as sf
import gradio as gr
import tempfile, os

MODEL_ID = 'openbmb/VoxCPM2'
model = VoxCPM.from_pretrained(MODEL_ID, load_denoiser=False)
SR = int(getattr(model.tts_model, 'sample_rate', 48000))
print('Model ready. Sample rate:', SR)

In [ ]:
def myanmar_only(text):
    return any('\u1000' <= c <= '\u109f' or '\uaa60' <= c <= '\uaa7f' for c in text)

def clone_voice(text, reference_audio, speed):
    text = (text or '').strip()
    if not text:
        raise gr.Error('မြန်မာစာသားထည့်ပါ။')
    if len(text) > 5000:
        raise gr.Error('စာသား 5,000 characters ထက် မကျော်ရပါ။')
    if not myanmar_only(text):
        raise gr.Error('Myanmar စာသားပဲ ထည့်ပါ။ English-only text မထောက်ပံ့ပါ။')
    if reference_audio is None:
        raise gr.Error('Reference voice audio ထည့်ပါ။')
    wav = model.generate(text=text, reference_wav_path=reference_audio, cfg_value=2.0, inference_timesteps=10)
    audio = wav.detach().float().cpu().numpy() if hasattr(wav, 'detach') else np.asarray(wav)
    audio = np.squeeze(audio)
    speed = max(0.5, min(2.0, float(speed)))
    if abs(speed - 1.0) > 0.001:
        audio = librosa.effects.time_stretch(audio, rate=speed)
    out = tempfile.NamedTemporaryFile(suffix='.wav', delete=False)
    out.close()
    sf.write(out.name, audio, SR)
    return out.name

with gr.Blocks(title='AI Voice Studio • Myanmar') as demo:
    gr.Markdown('# 🇲🇲 AI Voice Studio\n**Myanmar Voice Clone — VoxCPM2**')
    text = gr.Textbox(label='Myanmar Text', placeholder='မြန်မာစာသားရေးပါ...', lines=8, max_length=5000)
    ref = gr.Audio(label='Reference Voice', type='filepath')
    speed = gr.Slider(0.5, 2.0, value=1.0, step=0.05, label='Speed')
    run = gr.Button('▶ Myanmar Voice Clone', variant='primary')
    out = gr.Audio(label='Cloned Myanmar Voice', type='filepath')
    run.click(clone_voice, [text, ref, speed], out)

demo.launch(share=True, debug=False)